In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/Cinema_Audience_Forecasting_challenge/movie_theater_id_relation/movie_theater_id_relation.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/date_info/date_info.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_theaters/booknow_theaters.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_booking/cinePOS_booking.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_theaters/cinePOS_theaters.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_visits/booknow_visits.csv
/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv


In [5]:
# =============================================================================
# PART 3: FEATURE ENGINEERING (Creating New Features)
# =============================================================================
print("\nPART 3: Creating Features for Machine Learning")
print("=" * 70)

# Step 1: Merge all datasets together
print("Merging datasets...")
df = visits.merge(dates, on='show_date', how='left')
df = df.merge(theaters, on='book_theater_id', how='left') 

# Step 2: Create date-based features
print("Creating date features...")
df['day_of_week_num'] = df['show_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['month'] = df['show_date'].dt.month
df['day_of_month'] = df['show_date'].dt.day
df['is_weekend'] = (df['day_of_week_num'] >= 5).astype(int)  # 1 if weekend, 0 if weekday
df['quarter'] = df['show_date'].dt.quarter
df['week_of_year'] = df['show_date'].dt.isocalendar().week.astype(int)

# Sort by theater and date (important for next features)
df = df.sort_values(['book_theater_id', 'show_date'])

# Step 3: Calculate theater statistics (VERY IMPORTANT!)
print("Calculating theater statistics...")
theater_stats = visits.groupby('book_theater_id')['audience_count'].agg([
    'mean',   # Average audience for this theater
    'median', # Middle value
    'std',    # How much it varies
    'min',    # Minimum audience
    'max',    # Maximum audience
    'count'   # Number of records
]).reset_index()

# Rename columns for clarity
theater_stats.columns = ['book_theater_id', 'theater_mean', 'theater_median', 
                         'theater_std', 'theater_min', 'theater_max', 'theater_count']

# Add theater statistics to main dataframe
df = df.merge(theater_stats, on='book_theater_id', how='left')

# Step 4: Create lag features (previous days' data)
print("Creating lag features (past data)...")
# Lag_1 = audience count 1 day ago
df['lag_1'] = df.groupby('book_theater_id')['audience_count'].shift(1)
# Lag_3 = audience count 3 days ago
df['lag_3'] = df.groupby('book_theater_id')['audience_count'].shift(3)
# Lag_7 = audience count 7 days ago (last week)
df['lag_7'] = df.groupby('book_theater_id')['audience_count'].shift(7)
# Lag_14 = audience count 14 days ago (2 weeks ago)
df['lag_14'] = df.groupby('book_theater_id')['audience_count'].shift(14)

# Step 5: Create rolling statistics (moving averages)
print("Creating rolling statistics...")
# Average of last 3 days
df['rolling_mean_3'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
# Average of last 7 days
df['rolling_mean_7'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)
# Average of last 14 days
df['rolling_mean_14'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=14, min_periods=1).mean()
)
# Standard deviation of last 7 days
df['rolling_std_7'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()
)
# Standard deviation of last 14 days
df['rolling_std_14'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=14, min_periods=1).std()
)

# Step 6: Add booking data features
print("Adding booking information...")
booking['show_date'] = booking['show_datetime'].dt.date
booking['show_date'] = pd.to_datetime(booking['show_date'])

# Calculate total tickets and bookings per theater per day
booking_summary = booking.groupby(['book_theater_id', 'show_date']).agg({
    'tickets_booked': ['sum', 'mean', 'count']
}).reset_index()
booking_summary.columns = ['book_theater_id', 'show_date', 'total_tickets', 'avg_tickets', 'booking_count']

# Add booking data to main dataframe
df = df.merge(booking_summary, on=['book_theater_id', 'show_date'], how='left')

# Step 7: Encode categorical variables (convert text to numbers)
print("Converting categories to numbers...")
label_theater = LabelEncoder()
label_area = LabelEncoder()
df['theater_type_encoded'] = label_theater.fit_transform(df['theater_type'].astype(str))
df['theater_area_encoded'] = label_area.fit_transform(df['theater_area'].astype(str))

# Step 8: Handle missing values
print("Filling missing values...")
# Fill booking data missing values with 0
df['total_tickets'] = df['total_tickets'].fillna(0)
df['avg_tickets'] = df['avg_tickets'].fillna(0)
df['booking_count'] = df['booking_count'].fillna(0)

# Fill lag features with theater average
df['lag_1'] = df['lag_1'].fillna(df['theater_mean'])
df['lag_3'] = df['lag_3'].fillna(df['theater_mean'])
df['lag_7'] = df['lag_7'].fillna(df['theater_mean'])
df['lag_14'] = df['lag_14'].fillna(df['theater_mean'])

# Fill rolling statistics
df['rolling_mean_3'] = df['rolling_mean_3'].fillna(df['theater_mean'])
df['rolling_mean_7'] = df['rolling_mean_7'].fillna(df['theater_mean'])
df['rolling_mean_14'] = df['rolling_mean_14'].fillna(df['theater_mean'])
df['rolling_std_7'] = df['rolling_std_7'].fillna(df['theater_std'])
df['rolling_std_14'] = df['rolling_std_14'].fillna(df['theater_std'])

# Step 9: Select final features for model
feature_list = [
    'theater_mean', 'theater_median', 'theater_std',    # Theater statistics
    'day_of_week_num', 'month', 'is_weekend',           # Date features
    'lag_1', 'lag_7', 'lag_14',                         # Past data
    'rolling_mean_7', 'rolling_std_7',                  # Moving averages
    'total_tickets', 'booking_count',                   # Booking info
    'theater_type_encoded', 'theater_area_encoded'      # Category features
]

print(f"✅ Created {len(feature_list)} features")



PART 3: Creating Features for Machine Learning
Merging datasets...
Creating date features...
Calculating theater statistics...
Creating lag features (past data)...
Creating rolling statistics...
Adding booking information...
Converting categories to numbers...
Filling missing values...
✅ Created 15 features


## Part 3: Feature Engineering

In this part, we create new features to improve the predictive power of our models. Feature engineering steps include:

1. **Merging Datasets:** Combine visits, dates, and theaters data into a single dataframe.
2. **Date-Based Features:** Extract information from show dates such as:

   * Day of the week (`day_of_week_num`)
   * Month
   * Day of the month
   * Quarter
   * Week of the year
   * Weekend indicator (`is_weekend`)
3. **Theater Statistics:** Calculate per-theater statistics:

   * Mean, median, standard deviation, min, max audience
   * Total number of records per theater
4. **Lag Features:** Include past audience counts to capture trends:

   * 1-day, 3-day, 7-day, and 14-day lags
5. **Rolling Statistics:** Calculate moving averages and standard deviations:

   * Rolling mean (3, 7, 14 days)
   * Rolling standard deviation (7, 14 days)
6. **Booking Data Features:** Aggregate total tickets, average tickets, and booking counts per theater per day.
7. **Categorical Encoding:** Convert `theater_type` and `theater_area` to numerical values using Label Encoding.
8. **Handling Missing Values:** Fill missing lag, rolling, and booking data with meaningful defaults.
9. **Final Feature Selection:** Select a set of **15 important features** for machine learning models.

These engineered features help the model capture temporal trends, theater-specific patterns, and booking behavior.


In [6]:
# =============================================================================
# PART 4: PREPARING DATA FOR TRAINING
# =============================================================================
print("\nPART 4: Preparing Data for Model Training")
print("=" * 70)

# X = features (input), y = target (output to predict)
X = df[feature_list].fillna(0)
y = df['audience_count']

# Remove any infinite values
X = X.replace([np.inf, -np.inf], 0)

# Split data: 80% for training, 20% for validation
split_point = int(0.8 * len(df))
X_train = X[:split_point]
X_val = X[split_point:]
y_train = y[:split_point]
y_val = y[split_point:]

print(f"Training data size: {X_train.shape}")
print(f"Validation data size: {X_val.shape}")


PART 4: Preparing Data for Model Training
Training data size: (171236, 15)
Validation data size: (42810, 15)


## Part 4: Preparing Data for Model Training

In this part, we prepare the data for machine learning by performing the following steps:

1. **Define Features and Target:**

   * `X` contains the **input features** selected in Part 3.
   * `y` contains the **target variable**, which is `audience_count`.

2. **Handle Missing and Infinite Values:**

   * Replace any missing values with `0`.
   * Remove infinite values to prevent training issues.

3. **Train-Validation Split:**

   * Split the dataset into **80% training** and **20% validation**.
   * `X_train`, `y_train` → used to train models.
   * `X_val`, `y_val` → used to evaluate model performance.

This ensures the models are trained on historical data and validated on unseen data for proper performance assessment.
